In [ ]:
import httpx
import time
from pathlib import Path
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma


class DocumentLoader:
    def __init__(self, folder: str = "docs"):
        self.folder = Path(folder)
        self.folder.mkdir(exist_ok=True)

    def load_pdf(self) -> List[Document]:
        """Load all PDFs"""
        docs = []
        pdf_files = list(self.folder.glob("*.pdf"))

        print(f"Found {len(pdf_files)} PDFs.")

        for pdf in pdf_files:
            loader = PyPDFLoader(str(pdf))
            pages = loader.load()

            #enrich metadata on each page
            for page in pages:
                page.metadata['source_file'] = pdf.name
                page.metadata['total_pages'] = len(pages)

            docs.extend(pages)

        return docs
    
    #### Text Chunking
    def chunk_documents(self, docs: List[Document]) -> List[Document]:
        """Split into Chunks"""

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=750,
            chunk_overlap=100,
            length_function=len,
            separators=["\n\n", "\n", ".", " ", ""]
        )   

        chunks = splitter.split_documents(docs)

        #filter out empty/junk chunks

        chunks = [
            c for c in chunks
            if len(c.page_content.strip()) > 50            #skip near empty chunks
            and c.page_content.count("\n") < 10             #too many new lines = header
            and not c.page_content.strip().isdigit()        # pure page number
        ]

        print(f"Created {len(chunks)} clean chunks")
        return chunks


In [ ]:
# #Test

# if __name__ == "__main__":
#     loader = DocumentLoader()
#     docs = loader.load_pdf()
#     chunks = loader.chunk_documents(docs)

#     print(f"First chunk : {chunks[0].page_content}")

## Vector Store

In [5]:
class VectorStore:

    def __init__(self, model: str = "nomic-embed-text", db_path: str = "data/chroma"):

        #ollama heath check
        self._check_ollama()
        self.embeddings = OllamaEmbeddings(model=model)
        self.db_path = Path(db_path)
        self.db_path.mkdir(parents=True, exist_ok=True)
        self.vectorstore = None  # store reference for reuse


    def _check_ollama(self):
        """check ollama is running and responsive"""
        try:
            resp = httpx.get("http://localhost:11434", timeout=3)
            resp.raise_for_status()
        except httpx.ConnectError:
            raise RuntimeError("Ollama is not running. Start it with : ollama serve")
        
        except httpx.HTTPStatusError as e:
            raise RuntimeError(f"Ollama returned as error : {e}.")


    def create(self, chunks, batch_size: int = 500):
        """create vector database"""
        print(f"Creating embeddings for {len(chunks)} chunks...")

        if not chunks:
            raise ValueError("No chunks provided.")
        
        start = time.time()
        self.vectorstore = None

        #calculate batches
        num_batches = -(-len(chunks) // batch_size)

        with tqdm(total=num_batches, desc="Embedding batches") as pbar:
            for i in range(0, len(chunks), batch_size):
                batch = chunks[i : i + batch_size]
                #print(f" Embedding batch {i // batch_size + 1} / {-(-len(chunks) // batch_size)}...")

                if self.vectorstore is None:
                    self.vectorstore =  Chroma.from_documents(
                                            documents=batch,
                                            embedding=self.embeddings,
                                            persist_directory=str(self.db_path),
                                            collection_name="rag_docs",
                                            collection_metadata={"hnsw:space": "cosine"}
                                        )
                else:
                    self.vectorstore.add_documents(batch)
                
                pbar.update(1)


        elapsed = round(time.time() - start, 2)
        print(f"Vector Store created in {elapsed}s with {len(chunks)} chunks.")
        return self.vectorstore
    
    def load(self):
        """"Load existing Vector Store"""
        if not any(self.db_path.iterdir()):
            raise RuntimeError(
                f"No vector store found at {self.db_path}. Run create first."
            )

        self.vectorstore = Chroma(
            persist_directory=str(self.db_path),
            embedding_function=self.embeddings
        )

        print(f"Loaded vector store from {self.db_path}")
        return self.vectorstore
    
    def search(self, query: str, k: int = 3, with_scores: bool = False):
        """Unified search method with optional relevance scores"""
        if self.vectorstore is None:
                self.vectorstore = self.load()

        if with_scores:
            return self.vectorstore.similarity_search_with_score(query, k=k)
        return self.vectorstore.similarity_search(query, k=k)
    

    def get_stats(self):
        """Get statistics of vactor database"""

        if self.vectorstore is None:
            self.vectorstore = self.load()

        try:
            collection = self.vectorstore._collection
            count = collection.count()

            return {
                'total_chunks': count,
                'db_path': str(self.db_path),
                'embedding_model': self.embeddings.model
            }
        except Exception as e:
            return {'error' : str(e)}
    

    def delete(self):
        """Wipe vector store to rebuild from scratch"""

        import shutil
        if self.db_path.exists():
            shutil.rmtree(self.db_path)
            self.db_path.mkdir(parents=True, exist_ok=True)
            self.vectorstore = None
            print("Vector Store deleted. ")

#### Testing new searching chunking approach

In [ ]:
# loader = DocumentLoader()
# docs = loader.load_pdf()
# chunks = loader.chunk_documents(docs)
# store = VectorStore()
# store.create(chunks)
# store.load()

# # Search with scores to tune k
# results = store.search("What is machine learning?", k=5, with_scores=True)

# #results = store.search("What is RAG?", k=3, with_scores=True)
# for doc, score in results:
#     print(f"Score: {score:.3f} | {doc.page_content[:60]}")


In [ ]:
# results = store.search("What is Deep Neural Network?", k=3, with_scores=False)

# #results = store.search("What is RAG?", k=3, with_scores=True)
# for doc, score in results:
#     print(f"Score: {score:.3f} | {doc.page_content[:100]}")

### Delete and Rebuild vector store

In [6]:
#Test

if __name__ == "__main__":
    #Load and Chunk
    loader = DocumentLoader()
    docs = loader.load_pdf()
    chunks = loader.chunk_documents(docs)

    print(f"First chunk : {chunks[0].page_content}")

    #create vactor store
    vs = VectorStore()
    # vs.delete() 
    vectorstore = vs.create(chunks)
    stats = vs.get_stats()

    print(f"\n📊 Vector Store Statistice: ")
    print(f"    Total chunks: {stats['total_chunks']}")
    print(f"    Databs=ase: {stats['db_path']}")
    print(f"    Model: {stats['embedding_model']}")


    #test search
    results = vectorstore.similarity_search("Machine Learning", k=2)
    print(f"Search results: {len(results)}")
    print(f"Top results: {results[0].page_content}")

Found 5 PDFs.
Created 2001 clean chunks
First chunk : and evaluation to large-scale deployment and operation.
—Andrei Lopatenko, Director Search and AI, Neuron7
This book serves as an essential guide for building AI products that can scale.
Unlike other books that focus on tools or current trends that are constantly
changing, Chip delivers timeless foundational knowledge. Whether you’re
a product manager or an engineer, this book effectively bridges the
collaboration gap between cross-functional teams, making it
a must-read for anyone involved in AI development.
—Aileen Bui, AI Product Operations Manager, Google


RuntimeError: Ollama is not running. Start it with : ollama serve

In [ ]:
vs.get_stats()

In [ ]:
results[0].page_content

### Test Search Quality

In [ ]:
def test_search():
    vs = VectorStore()
    vectorstore = vs.load()
    stats = vs.get_stats()

    print(f"\n📊 Vector Store Statistice: ")
    print(f"    Total chunks: {stats['total_chunks']}")
    print(f"    Databs=ase: {stats['db_path']}")
    print(f"    Model: {stats['embedding_model']}")


    test_queries = ["What is AI?",
                    "What is Machine Learning?",
                    "What is deep learning?",
                    "What is LSTM?"
                    ]
    


    for query in test_queries:
        print(f"\n{'='*60}")
        print(f"\n🔍 Query: {query}")
        print('='*60)
        results = vectorstore.similarity_search_with_score(query, k=3)

        for doc, score in results:
            print(f"Score: {score:.3f}")
            print(f" Source: {doc.metadata.get('page', 'N/A')}")
            print(f"Content: {doc.page_content[:150]}...")

In [ ]:
test_search()